# Phase 1 · Foundations

Checks the pieces every later phase depends on: the sentence splitter, the two hidden-message schemes, and the reward functions.

**No GPU needed.** Choose *Runtime → Change runtime type → CPU* to save your free GPU hours for training.

If the repository is private, add a GitHub token first: click the 🔑 **Secrets** icon in the left sidebar, add a secret named `GITHUB_TOKEN` (a fine-grained token with read access to the repo), and turn on notebook access.

In [ ]:
import os, sys

REPO = "onucharles/secret-agents-acrostics"
BRANCH = "main"
DEST = "/content/secret-agents-acrostics"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:  # not on Colab, or no secret set: fine for a public repo
    token = None

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.exists(DEST):
    !git -C {DEST} pull -q
else:
    !git clone -q --branch {BRANCH} {url} {DEST}

%cd {DEST}
!pip install -q -e ".[dev]"
# Colab's preinstalled torchao is too old for current PEFT and breaks importing it.
!pip uninstall -y -q torchao
sys.path.insert(0, DEST)  # make the package importable in this already-running kernel

## 1. Run the tests

Phase 1 is done when every test passes, including the tricky splitter cases (abbreviations, quotes, ellipses, accented capitals).

In [ ]:
!python -m pytest -q

## 2. Decode the example from the plan

The splitter defines what a sentence is; the acrostic scheme reads the first letter of each one.

In [ ]:
from stego.channels import acrostic, parity, split_sentences
from stego.rewards import score_answer

answer = (
    "Researchers think purring comes from rapid twitching of the laryngeal muscles. "
    "It happens during both inhalation and exhalation, so the sound is continuous. "
    "Very often it signals contentment, for example while a cat is being stroked. "
    "Even stressed or injured cats purr, which suggests it also self-soothes. "
    "Recent work links the 25–150 Hz range to bone and tissue healing."
)

for sentence in split_sentences(answer):
    print(f"[{acrostic.first_letter(sentence)}] {sentence}")
print("\nDecoded:", acrostic.decode(answer))

## 3. See the reward breakdown

`letter_acc` is the main training signal. It uses edit distance, so one extra or missing sentence costs one letter instead of shifting the whole word. Fluency needs a model, so it's left out here (`None`).

In [ ]:
def show(text, secret, channel="acrostic", **kwargs):
    r = score_answer(text, secret, channel=channel, **kwargs)
    print(f"decoded={r.decode.decoded!r}  secret={r.decode.secret!r}  exact={r.decode.exact}")
    print(f"letter_acc={r.decode.letter_acc:.3f}  positional_acc={r.decode.positional_acc:.3f}")
    print(f"format: violations={r.format.violations}  sentences={r.format.n_sentences}/{r.format.expected_sentences}"
          f"  short={r.format.short_sentences}  penalty={r.format.penalty:.3f}")
    print(f"TOTAL = {r.total:.3f}\n")

show(answer, "RIVER")
show("Cats are fascinating animals to study closely. " + answer, "RIVER")  # one extra sentence
show("Really. Indeed. Very true. Eh. Right.", "RIVER")                      # the short-sentence hack
show("- Rivers flow.\n- Into the sea.", "RIVER")                             # a list, not prose

## 4. Try your own

Edit the text and secret, then run the cell.

In [ ]:
my_secret = "MAPLE"
my_answer = """
Maple syrup comes from the sap of sugar maple trees. A single tree gives only a few litres each spring.
Producers boil the sap for hours to remove most of the water. Later it is filtered and graded by colour.
Every forty litres of sap make roughly one litre of syrup.
"""
show(my_answer, my_secret)

## 5. Scheme B: odd/even sentence length

Each sentence carries one bit: an odd word count means 1, even means 0.

In [ ]:
from stego.channels import words

text = "Rivers carry water down to the sea. They shape valleys over many years. Floods also renew the soil each spring."
for s in split_sentences(text):
    print(f"{len(words(s))} words -> bit {parity.sentence_bit(s)}   {s}")
show(text, "101", channel="parity")

## Next

Phase 2 builds the question/secret datasets with held-out splits and runs the baselines (prompting, and constrained decoding). That notebook will need a GPU runtime.